In [1]:
import time
import random
import numpy as np
import pandas as pd
import torch
from torch import nn
from scipy.optimize import linprog

# ============================================================
# 1. Reproducibility and device
# ============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

# ============================================================
# 2. Objective network
# Paper: max z = -2*x1 - x2
# Equivalent minimization used by NOM: f = 2*x1 + x2
# ============================================================
class ObjectiveNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(2, 1, bias=False)
        with torch.no_grad():
            self.linear.weight.copy_(
                torch.tensor([[2.0, 1.0]], dtype=torch.float32)
            )
        for p in self.parameters():
            p.requires_grad_(False)

    def forward(self, x):
        return self.linear(x)


obj_net = ObjectiveNet().to(device).eval()


def true_objective(x):
    return 2.0 * x[..., 0] + x[..., 1]


# Exact solution of the continuous semi-infinite problem
X_EXACT = np.array([1.0 / 9.0, 4.0 / 9.0], dtype=np.float64)
F_EXACT = 2.0 / 3.0

# ============================================================
# 3. Discretized constraints
# -t*x1 + (t-1)*x2 <= t^2 - t, t in [0,1]
# plus x1>=0, x2>=0 -> -x1<=0, -x2<=0
# ============================================================
def build_constraints(m):
    if m < 2:
        raise ValueError("m must be >= 2")

    t = torch.linspace(0.0, 1.0, m, dtype=torch.float32, device=device)
    A_t = torch.column_stack((-t, t - 1.0))
    b_t = t.square() - t

    A_nonneg = torch.tensor(
        [[-1.0, 0.0], [0.0, -1.0]], dtype=torch.float32, device=device
    )
    b_nonneg = torch.zeros(2, dtype=torch.float32, device=device)

    A = torch.cat((A_t, A_nonneg), dim=0)
    b = torch.cat((b_t, b_nonneg), dim=0)
    return t, A, b


# ============================================================
# 4. Corrected Neural Optimization Machine
# ============================================================
class NeuralOptimizationMachine(nn.Module):
    def __init__(self, obj_net, A, b, initial_points, penalty=20.0):
        super().__init__()
        self.obj_net = obj_net
        self.register_buffer("A", A.detach().clone())
        self.register_buffer("b", b.detach().clone())
        self.penalty = float(penalty)

        # Each row is one independent starting point / candidate solution.
        self.x = nn.Parameter(
            torch.as_tensor(initial_points, dtype=torch.float32, device=A.device)
        )

    def forward(self):
        objective = self.obj_net(self.x)                         # [n_starts,1]
        raw_violation = self.x @ self.A.T - self.b              # [n_starts,n_constraints]
        violation = torch.relu(raw_violation)

        # IMPORTANT FIX:
        # Using SUM(violation) makes the penalty grow with m and changes the
        # optimization problem when m goes 11 -> 101 -> 1001 -> 10001.
        # Worst-constraint penalty has a stable meaning for every grid size.
        max_violation = violation.max(dim=1, keepdim=True).values

        per_start_loss = objective + self.penalty * max_violation
        return per_start_loss.mean(), objective, violation


# ============================================================
# 5. Check against the ORIGINAL continuous constraint family
# ============================================================
def dense_constraint_check(x, n_dense=100_001):
    x1, x2 = float(x[0]), float(x[1])
    t = np.linspace(0.0, 1.0, n_dense, dtype=np.float64)

    lhs = -t * x1 + (t - 1.0) * x2
    rhs = t**2 - t
    violation = lhs - rhs  # <= 0 is feasible

    idx = int(np.argmax(violation))
    return float(violation[idx]), float(t[idx])


# ============================================================
# 6. Exact finite-LP benchmark for a chosen discretization
# ============================================================
def solve_finite_lp(m):
    # Use float64 numpy here so the benchmark is not limited by float32.
    t = np.linspace(0.0, 1.0, m, dtype=np.float64)
    A_t = np.column_stack((-t, t - 1.0))
    b_t = t**2 - t

    result = linprog(
        c=np.array([2.0, 1.0], dtype=np.float64),
        A_ub=A_t,
        b_ub=b_t,
        bounds=[(0.0, None), (0.0, None)],
        method="highs",
    )
    if not result.success:
        raise RuntimeError(f"linprog failed for m={m}: {result.message}")
    return result.x, float(result.fun)


# ============================================================
# 7. NOM solver for one grid size
# ============================================================
def solve_nom(
    m,
    n_starts=5,
    epochs=10_000,
    lr=1e-3,
    penalty=20.0,
    feasible_tol=1e-6,
    verbose=False,
):
    _, A, b = build_constraints(m)

    rng = np.random.default_rng(SEED + m)
    initial_points = rng.uniform(0.0, 1.0, size=(n_starts, 2))

    nom = NeuralOptimizationMachine(
        obj_net=obj_net,
        A=A,
        b=b,
        initial_points=initial_points,
        penalty=penalty,
    ).to(device)

    optimizer = torch.optim.Adam([nom.x], lr=lr)

    # Best GRID-feasible iterate for every start.
    best_obj = torch.full((n_starts,), float("inf"), device=device)
    best_x = torch.zeros((n_starts, 2), dtype=torch.float32, device=device)
    found_feasible = torch.zeros(n_starts, dtype=torch.bool, device=device)

    start_time = time.perf_counter()

    for epoch in range(epochs):
        optimizer.zero_grad(set_to_none=True)
        loss, _, _ = nom()
        loss.backward()
        optimizer.step()

        # Recompute metrics AFTER the optimizer update.
        with torch.no_grad():
            grid_violation = torch.relu(nom.x @ A.T - b).max(dim=1).values
            objective_now = true_objective(nom.x)

            better = (grid_violation <= feasible_tol) & (objective_now < best_obj)
            if better.any():
                best_obj[better] = objective_now[better]
                best_x[better] = nom.x[better]
                found_feasible[better] = True

        if verbose and (epoch % 3000 == 0 or epoch == epochs - 1):
            print(
                f"m={m:5d} epoch={epoch:5d} "
                f"mean_loss={loss.item():.8f} "
                f"best_feasible={found_feasible.sum().item()}/{n_starts}"
            )

    elapsed = time.perf_counter() - start_time

    # If a start never crossed the strict tolerance, use its final iterate.
    with torch.no_grad():
        final_grid_violation = torch.relu(nom.x @ A.T - b).max(dim=1).values
        final_obj = true_objective(nom.x)

        for i in range(n_starts):
            if not found_feasible[i]:
                best_x[i] = nom.x[i]
                best_obj[i] = final_obj[i]

    # Choose the best grid-feasible start; if none, choose least violating start.
    candidates = []
    start_rows = []

    for i in range(n_starts):
        x_np = best_x[i].detach().cpu().numpy().astype(np.float64)
        with torch.no_grad():
            x_t = best_x[i].unsqueeze(0)
            grid_v = torch.relu(x_t @ A.T - b).max().item()
            f_val = float(true_objective(x_t).item())

        dense_v, worst_t = dense_constraint_check(x_np)
        grid_feasible = grid_v <= feasible_tol

        row = {
            "m": m,
            "start": i + 1,
            "x1": x_np[0],
            "x2": x_np[1],
            "objective": f_val,
            "grid_max_violation": grid_v,
            "dense_max_violation": dense_v,
            "worst_t": worst_t,
        }
        start_rows.append(row)

        # Grid feasibility is the correct criterion here because NOM is solving
        # the discretized LP. Dense violation is reported separately to measure
        # discretization error.
        rank = (0 if grid_feasible else 1, f_val if grid_feasible else grid_v, f_val)
        candidates.append((rank, row))

    winner = min(candidates, key=lambda item: item[0])[1]
    lp_x, lp_f = solve_finite_lp(m)

    result_row = {
        "m": m,
        "nom_x1": winner["x1"],
        "nom_x2": winner["x2"],
        "nom_objective": winner["objective"],
        "nom_grid_max_violation": winner["grid_max_violation"],
        "nom_dense_max_violation": winner["dense_max_violation"],
        "nom_worst_t": winner["worst_t"],
        "lp_x1": lp_x[0],
        "lp_x2": lp_x[1],
        "lp_objective": lp_f,
        "objective_error_vs_finite_lp": abs(winner["objective"] - lp_f),
        "objective_error_vs_exact_lsip": abs(winner["objective"] - F_EXACT),
        "x_error_vs_exact_lsip": float(
            np.linalg.norm(np.array([winner["x1"], winner["x2"]]) - X_EXACT)
        ),
        "iterations": epochs,
        "converged": bool(winner["grid_max_violation"] <= feasible_tol),
        "runtime_sec": elapsed,
    }

    return result_row, pd.DataFrame(start_rows)


# ============================================================
# 8. Run the paper-style grid refinement experiment
# ============================================================
if __name__ == "__main__":
    grid_sizes = [11, 101, 1001, 10001]

    print("\nExact continuous LSIP solution")
    print(f"x1* = {X_EXACT[0]:.9f}")
    print(f"x2* = {X_EXACT[1]:.9f}")
    print(f"min(2*x1+x2) = {F_EXACT:.9f}")
    print(f"original max(-2*x1-x2) = {-F_EXACT:.9f}\n")

    results = []
    all_starts = []

    for m in grid_sizes:
        result, starts = solve_nom(
            m=m,
            n_starts=5,
            epochs=10_000,
            lr=1e-3,
            penalty=20.0,
            feasible_tol=1e-6,
            verbose=False,
        )
        results.append(result)
        all_starts.append(starts)

        print(
            f"m={m:5d} | "
            f"NOM x=({result['nom_x1']:.9f}, {result['nom_x2']:.9f}) | "
            f"f={result['nom_objective']:.9f} | "
            f"grid viol={result['nom_grid_max_violation']:.2e} | "
            f"dense viol={result['nom_dense_max_violation']:.2e} | "
            f"finite-LP f={result['lp_objective']:.9f}"
        )

    results_df = pd.DataFrame(results)
    starts_df = pd.concat(all_starts, ignore_index=True)

    results_path = "nom_lsip_results.csv"
    starts_path = "nom_lsip_all_starts.csv"
    results_df.to_csv(results_path, index=False)
    starts_df.to_csv(starts_path, index=False)

    print("\nFinal comparison")
    print(results_df.to_string(index=False))
    print(f"\nSaved: {results_path}")
    print(f"Saved: {starts_path}")


Running on: cuda

Exact continuous LSIP solution
x1* = 0.111111111
x2* = 0.444444444
min(2*x1+x2) = 0.666666667
original max(-2*x1-x2) = -0.666666667

m=   11 | NOM x=(0.119998693, 0.419999927) | f=0.659997344 | grid viol=9.39e-07 | dense viol=2.50e-03 | finite-LP f=0.660000000
m=  101 | NOM x=(0.112204112, 0.442189127) | f=0.666597366 | grid viol=9.69e-07 | dense viol=2.59e-05 | finite-LP f=0.666600000
m= 1001 | NOM x=(0.111208424, 0.444246292) | f=0.666663170 | grid viol=9.54e-07 | dense viol=1.20e-06 | finite-LP f=0.666666000
m=10001 | NOM x=(0.111300819, 0.444062412) | f=0.666664064 | grid viol=9.83e-07 | dense viol=9.54e-07 | finite-LP f=0.666666600

Final comparison
    m   nom_x1   nom_x2  nom_objective  nom_grid_max_violation  nom_dense_max_violation  nom_worst_t    lp_x1    lp_x2  lp_objective  objective_error_vs_finite_lp  objective_error_vs_exact_lsip  x_error_vs_exact_lsip  iterations  converged  runtime_sec
   11 0.119999 0.420000       0.659997            9.387732e-07    